In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
!pip install -q torch torchvision matplotlib scikit-learn kaggle kagglehub pillow

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import sklearn
import kagglehub
from PIL import Image

print("Libraries loaded successfully!")

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "ashishsaxena2209/animal-image-datasetdog-cat-and-panda"
)

print("Dataset downloaded!")
print(dataset_path)

In [ ]:
import os

animals_dir = None

for root, dirs, files in os.walk(dataset_path):
    if "cats" in dirs and "dogs" in dirs and "panda" in dirs:
        animals_dir = root
        break

print("Correct dataset folder:", animals_dir)

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(animals_dir, transform=transform)

print("Classes:", dataset.classes)
print("Number of images:", len(dataset))
print("Class mapping:", dataset.class_to_idx)

In [ ]:
print(os.listdir(animals_dir))

In [ ]:
print("Cats:", len(os.listdir(os.path.join(animals_dir, "cats"))))
print("Dogs:", len(os.listdir(os.path.join(animals_dir, "dogs"))))
print("Pandas:", len(os.listdir(os.path.join(animals_dir, "panda"))))

In [ ]:
from torchvision import datasets, transforms

class_names = ["cats", "dogs", "panda"]

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(
    animals_dir,
    transform=base_transform
)

valid_samples = [
    (path, class_names.index(os.path.basename(os.path.dirname(path))))
    for path, label in full_dataset.samples
    if os.path.basename(os.path.dirname(path)) in class_names
]

full_dataset.samples = valid_samples
full_dataset.targets = [label for _, label in valid_samples]
full_dataset.classes = class_names
full_dataset.class_to_idx = {
    "cats": 0,
    "dogs": 1,
    "panda": 2
}

print("Classes:", full_dataset.classes)
print("Total images:", len(full_dataset))
print("Class mapping:", full_dataset.class_to_idx)

In [ ]:
from torch.utils.data import random_split

total = len(full_dataset)

train_size = int(0.70 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

print("DataLoaders created successfully!")

In [ ]:
images, labels = next(iter(train_loader))

print("Image shape:", images.shape)
print("Minimum label:", labels.min().item())
print("Maximum label:", labels.max().item())

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights="DEFAULT")

print("ResNet18 loaded successfully!")

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False

print("Convolution layers frozen.")

In [ ]:
num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 3)
)

model = model.to(device)

print(model.fc)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")

In [ ]:
num_epochs = 10

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

best_val_accuracy = 0.0

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.2f}%"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "best_cat_dog_panda_resnet18.pth"
        )

        print("Best model saved!")

In [ ]:
model.load_state_dict(
    torch.load(
        "best_cat_dog_panda_resnet18.pth",
        map_location=device
    )
)

model.eval()

print("Best model loaded successfully!")

In [ ]:
test_loss = 0.0
test_correct = 0
test_total = 0

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_accuracy = 100 * test_correct / test_total

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 2), "%")

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["cats", "dogs", "panda"]
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels,
    all_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["cats", "dogs", "panda"]
)

disp.plot()

plt.title("Cat Dog Panda Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Step 15: Example Predictions

model.eval()

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    predictions = torch.argmax(outputs, dim=1)

class_names = ["cats", "dogs", "panda"]

plt.figure(figsize=(12, 8))

for i in range(9):
    image = images[i].cpu().permute(1, 2, 0).numpy()

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image * std + mean
    image = np.clip(image, 0, 1)

    plt.subplot(3, 3, i + 1)
    plt.imshow(image)

    actual = class_names[labels[i].item()]
    predicted = class_names[predictions[i].item()]

    plt.title(f"Actual: {actual}\nPredicted: {predicted}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "cat_dog_panda_resnet18.pth"
)

print("Final model saved successfully!")

In [ ]:
import os

print(
    "Model file exists:",
    os.path.exists("cat_dog_panda_resnet18.pth")
)